<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ Cohere Transcribe</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Colab T4 Edition — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>State-of-the-art ASR · 14 Languages · Long-form Audio · 2B Parameter Conformer Model</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

## 🔐 Before You Start — Required Setup

This notebook uses **[CohereLabs/cohere-transcribe-03-2026](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026)**, a **gated model** on Hugging Face. You must complete the steps below before running any code cell.

### Step 1 — Accept the Model Terms on Hugging Face

> The model is gated — Cohere requires you to agree to usage terms before downloading.

1. Go to 👉 [huggingface.co/CohereLabs/cohere-transcribe-03-2026](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026)
2. Log in (or create a free account at [huggingface.co/join](https://huggingface.co/join))
3. Click **"Agree and access repository"**

### Step 2 — Create a Hugging Face Access Token

1. Go to 👉 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Click **"New token"** → name it anything → role: **Read**
3. Copy the token and paste it when the login cell prompts you

### Step 3 — Enable GPU & Run All

| Platform | Steps |
|---|---|
| **Google Colab** | Runtime → Change runtime type → **T4 GPU** → Run all |
| **Kaggle** | Settings → Accelerator → **GPU T4 x1** → Run all |

In [ ]:
#@title 📦 Install Dependencies
!pip install -q "transformers>=5.4.0" torch huggingface_hub soundfile librosa sentencepiece protobuf gradio

In [ ]:
#@title 🔐 Login to Hugging Face (required — gated model)
from huggingface_hub import login
login()

In [ ]:
#@title 🚀 Load Model (float16 — fits on T4 16GB VRAM)
import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration

model_id = "CohereLabs/cohere-transcribe-03-2026"

processor = AutoProcessor.from_pretrained(model_id)
model = CohereAsrForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✅ Model loaded successfully!")

In [ ]:
#@title 🎛️ Launch Gradio Interface
import gradio as gr
import librosa
import numpy as np
import time
from transformers.audio_utils import load_audio

# ── AIQUEST Branding ────────────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">⚡ Cohere Transcribe</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; 14 Languages · 2B Conformer ASR Model</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
# ────────────────────────────────────────────────────────────────────────────

LANGUAGES = [
    ("English", "en"), ("French", "fr"), ("German", "de"), ("Spanish", "es"),
    ("Portuguese", "pt"), ("Italian", "it"), ("Dutch", "nl"), ("Polish", "pl"),
    ("Greek", "el"), ("Arabic", "ar"), ("Japanese", "ja"), ("Korean", "ko"),
    ("Chinese", "zh"), ("Vietnamese", "vi"),
]
LANGUAGE_MAP = {code: label for label, code in LANGUAGES}

def transcribe(audio_input, language, punctuation):
    if audio_input is None:
        return "⚠️ Please upload an audio file or record from microphone."

    try:
        if isinstance(audio_input, tuple):
            sr, audio_data = audio_input
            audio_data = audio_data.astype(np.float32)
            if audio_data.ndim > 1:
                audio_data = audio_data.mean(axis=1)
            if audio_data.max() > 1.0:
                audio_data = audio_data / 32768.0
            if sr != 16000:
                audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)
                sr = 16000
        else:
            audio_data = load_audio(audio_input, sampling_rate=16000)
            sr = 16000

        duration = len(audio_data) / sr

        inputs = processor(
            audio_data,
            sampling_rate=sr,
            return_tensors="pt",
            language=language,
            punctuation=punctuation,
        )

        audio_chunk_index = inputs.pop("audio_chunk_index", None)
        inputs = inputs.to(model.device, dtype=model.dtype)

        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512)
        elapsed = time.time() - start

        if audio_chunk_index is not None:
            text = processor.decode(
                outputs,
                skip_special_tokens=True,
                audio_chunk_index=audio_chunk_index,
                language=language,
            )
            if isinstance(text, list):
                text = text[0] if len(text) == 1 else " ".join(text)
        else:
            text = processor.decode(outputs, skip_special_tokens=True)

        lang_label = LANGUAGE_MAP.get(language, language)
        rtfx = duration / elapsed if elapsed > 0 else 0
        info = f"⏱️ {elapsed:.1f}s | RTFx: {rtfx:.1f}x | Duration: {duration:.1f}s | {lang_label} ({language}) | Punctuation: {'✅' if punctuation else '❌'}"
        return f"{info}\n\n📝 Transcription:\n\n{text}"

    except Exception as e:
        return f"❌ Error: {str(e)}"


with gr.Blocks(theme=gr.themes.Soft(), css=CSS) as demo:

    gr.HTML(BRAND_HTML)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            gr.Markdown("### Audio Input")
            audio_input = gr.Audio(
                label="Record or upload audio",
                type="filepath",
                sources=["microphone", "upload"],
            )
            language = gr.Dropdown(
                label="Language",
                choices=LANGUAGES,
                value="en",
                info="Select the language spoken in the audio.",
            )
            punctuation = gr.Checkbox(value=True, label="Include Punctuation")
            with gr.Row():
                btn = gr.Button("🚀 Transcribe", variant="primary", size="lg")
                clear_btn = gr.Button("🔄 Reset", variant="secondary", size="lg")

        with gr.Column(scale=7):
            gr.Markdown("### Transcript")
            output = gr.Textbox(
                label="Result",
                lines=15,
                max_lines=25,
                show_copy_button=True,
                interactive=False,
                placeholder="The transcription will appear here.",
            )

    def reset_ui():
        return None, "en", True, ""

    btn.click(fn=transcribe, inputs=[audio_input, language, punctuation], outputs=output)
    clear_btn.click(fn=reset_ui, outputs=[audio_input, language, punctuation, output])

    gr.Markdown(
        "Model: [CohereLabs/cohere-transcribe-03-2026](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) | "
        "Apache 2.0 | 14 languages: en, fr, de, es, pt, it, nl, pl, el, ar, ja, ko, zh, vi"
    )

demo.launch(share=True, debug=True)

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---